<a href="https://colab.research.google.com/github/1PD-IS-NO-1/ALL-TEXT-SUMMARIZER-/blob/main/fine_Tuning_gpt2_on_textdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers peft datasets accelerate pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fss

In [2]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = ''
        for page in pdf.pages:
            text += page.extract_text()
    return text

# Extract text from two PDFs
pdf1_text = extract_text_from_pdf('/content/B6527129219.pdf')
pdf2_text = extract_text_from_pdf('/content/JETIR2210312.pdf')

# Combine the text
combined_text = pdf1_text + '\n' + pdf2_text

# Save the combined text to a file
with open('dataset.txt', 'w') as f:
    f.write(combined_text)


In [3]:
from datasets import Dataset

# Create a dataset from the text file
data = {"text": [line for line in combined_text.split('\n') if line.strip()]}
dataset = Dataset.from_dict(data)

# Split into train and validation sets
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset['train']
val_dataset = dataset['test']


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# Load GPT-2 and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)



# Ensure the tokenizer has a pad token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})
    model.resize_token_embeddings(len(tokenizer))  # Resize the model embeddings

def preprocess_data(examples):
    # Tokenize the input text
    tokenized = tokenizer(
        examples['text'], truncation=True, padding="max_length", max_length=512
    )
    # Use `input_ids` as `labels`
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Apply preprocessing to the datasets
train_dataset = train_dataset.map(preprocess_data, batched=True)
val_dataset = val_dataset.map(preprocess_data, batched=True)




from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    report_to="wandb",
)

from transformers import DataCollatorForLanguageModeling, Trainer

# Define the data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # Set mlm=True for masked language modeling
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,  # Use data_collator instead of tokenizer
)


trainer.train()
model.save_pretrained('./lora_gpt2')

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Map:   0%|          | 0/68 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,4.244900,3.578927
2,2.814500,3.345911
3,2.455000,3.282505
4,2.034400,3.295881
5,1.869700,3.307283


In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the tokenizer and LoRA fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained('./lora_gpt2')

# Move the model to the appropriate device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=450,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Test query
query = "Describe the approach in step-by-step how to detect a helmet using AI techniques."
print(generate_text(query))



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Describe the approach in step-by-step how to detect a helmet using AI techniques. The methodology used here, which was developed with input from members of our research team and industry stakeholders including: “Machine Learning” , Journal for Research In Applied Science & Engineering Technology (JETACT), ISSN 1474–9653; IC Value 9592 – Impact Factor 5·7(v5) = 0x89 · Volume 8 Issue 11 pp 67pp December 2020. DOI : 10 .17740/ijitee..0017&s27© 2022 Jetact International Academic Publication Number 201611075 ISBN 97814887125619 USP Title Helmet Detection by Using Convolutional Neural Networks Modeling Algorithms For Machine Vision And Image Processing UGC approved WITC Approved Peer Reviewed AND refereer Microsoft ACM License plate collection system is currently being proposed as an alternative methodologies [15]. Currently this model would work only if there are many riders riding on particular road at once so that it can be easily extracted without detection due its limited features such 

In [21]:
import shutil
from google.colab import files

# Path to the directory
directory_path = '/content/lora_gpt2'

# Output zip file name
output_zip = '/content/lora_gpt2.zip'

# Compress the directory
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', directory_path)

# Download the zip file
files.download(output_zip)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>